# SHIVA-LLM · Objective 02 · Stage 4A-C
## Reward and simulator redesign — assumption-labeled contextual bandit

**Runtime:** Google Colab CPU (Python 3), ~12–16 GB RAM.  
**Upload:** `objective02_stage4a_simulator.zip` and `objective02_stage4ab_robustness_audit.zip`.

This notebook **does not estimate causal effects of instructional actions**. It uses the Stage 4A calibrated *observational* response predictions and explicitly assumed effects of easy/medium/hard actions. Stage 4A-B showed fixed-easy dominance; this stage tests whether a mastery/challenge utility can produce state-dependent choices **without tuning the design on the test split**. Validation data is used for design diagnostics; test is a locked audit.

In [ ]:
import os, io, json, zipfile, pathlib, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

SEED=42
rng=np.random.default_rng(SEED)
WORK=pathlib.Path('/content/stage4ac')
WORK.mkdir(parents=True,exist_ok=True)
uploaded=files.upload()
required=['objective02_stage4a_simulator.zip','objective02_stage4ab_robustness_audit.zip']
missing=[x for x in required if x not in uploaded and not pathlib.Path(x).exists()]
assert not missing, f'Missing inputs: {missing}'
z4a=zipfile.ZipFile(required[0])
z4ab=zipfile.ZipFile(required[1])
assert z4a.testzip() is None and z4ab.testzip() is None
print('Stage 4A files:',z4a.namelist())
print('Stage 4A-B files:',z4ab.namelist())

In [ ]:
pred=pd.read_csv(io.BytesIO(z4a.read('response_model_predictions.csv')))
spec=json.loads(z4a.read('simulator_spec.json'))
audit=json.loads(z4ab.read('audit_report.json'))
assert {'split','user_id','order_id','observed_correct','p_observed'} <= set(pred.columns)
assert set(pred.split.unique())=={'train','val','test'}
assert not pred.duplicated(['user_id','order_id']).any()
assert pred.p_observed.between(0,1).all()
assert pred.observed_correct.isin([0,1]).all()
groups={s:set(pred.loc[pred.split==s,'user_id']) for s in ['train','val','test']}
assert not (groups['train']&groups['val'] or groups['train']&groups['test'] or groups['val']&groups['test'])
assert len(pred)==278942, f'Unexpected size: {len(pred)}'
assert not spec.get('counterfactual_effect_identified',False)
assert not audit.get('causal_action_effects_identified',False)
print(pred.groupby('split').agg(events=('user_id','size'),learners=('user_id','nunique')))
print('Stage 4A-B original moderate easy dominance:',audit.get('moderate_original_easy_optimal_fraction'))

### Assumptions and proposed reward

For each observed response probability \(p\), the **assumed** probability under action \(a\) is:

\[
q_a(p)=\sigma(\operatorname{logit}(p)+\delta_a),\quad
\delta=(+d,0,-d).
\]

The proposed *expected* utility is:

\[
U_a(p)=w_c q_a(p)+w_m q_a(p)c_a+
w_s\exp\!\left(-\frac{(q_a(p)-\tau)^2}{2h^2}\right)
-w_k k_a.
\]

Here \(c_a\) represents assumed challenge credit (not measured learning), \(\tau\) is the target success probability, \(h\) controls suitability width, and \(k_a\) is assumed effort cost. This reward **does not measure actual mastery or knowledge growth**.

All parameter values below are *predeclared exploratory scenarios*. They are not fitted from action-outcome data. A **zero-effect** scenario is essential: when \(d=0\), only assumed utility preferences can separate actions.

In [ ]:
ACTIONS=np.array(['easy','medium','hard'])
CHALLENGE=np.array([0.20,0.60,1.00]) # normative challenge credit, not estimated mastery
COST=np.array([0.15,0.35,0.60])      # assumed instructional cost
SCENARIOS={
 'zero':0.00, 'small':0.15, 'moderate':0.40, 'large':0.80
}
# Candidate utilities are specified independently of test outcomes.
DESIGNS={
 'original_like': {'wc':0.60,'wm':0.00,'ws':0.20,'wk':0.20,'tau':0.65,'h':0.18},
 'balanced':      {'wc':0.30,'wm':0.40,'ws':0.25,'wk':0.05,'tau':0.70,'h':0.20},
 'mastery_focus': {'wc':0.20,'wm':0.55,'ws':0.20,'wk':0.05,'tau':0.75,'h':0.20},
 'suitability':   {'wc':0.15,'wm':0.20,'ws':0.60,'wk':0.05,'tau':0.70,'h':0.14},
}
def sigmoid(x): return 1/(1+np.exp(-np.clip(x,-35,35)))
def action_probs(p,d):
    p=np.clip(np.asarray(p,dtype=float),1e-6,1-1e-6)
    logits=np.log(p/(1-p))
    return sigmoid(logits[:,None]+np.array([d,0,-d])[None,:])
def utilities(p,d,design):
    q=action_probs(p,d)
    g=DESIGNS[design]
    suit=np.exp(-0.5*((q-g['tau'])/g['h'])**2)
    u=g['wc']*q + g['wm']*q*CHALLENGE[None,:] + g['ws']*suit - g['wk']*COST[None,:]
    return q,u
def summarize(p,split,design,scenario):
    d=SCENARIOS[scenario]
    q,u=utilities(p,d,design)
    opt=u.argmax(axis=1)
    means=u.mean(axis=0)
    oracle=u.max(axis=1).mean()
    return {'split':split,'design':design,'scenario':scenario,'effect_d':d,'n':len(p),
        **{f'optimal_{a}_fraction':float((opt==i).mean()) for i,a in enumerate(ACTIONS)},
        **{f'fixed_{a}_mean_reward':float(means[i]) for i,a in enumerate(ACTIONS)},
        'oracle_mean_reward':float(oracle),
        'oracle_minus_best_fixed':float(oracle-means.max()),
        'best_fixed_action':str(ACTIONS[means.argmax()]),
        'mean_expected_success_under_oracle':float(q[np.arange(len(p)),opt].mean())}

### Validation-only exploration

We inspect whether any utility produces different preferred actions for different readiness values. **A nontrivial policy is not itself evidence of validity.** We explicitly report cases where the best action is constant.

In [ ]:
val=pred[pred.split=='val'].copy()
rows=[summarize(val.p_observed.to_numpy(),'val',design,scenario)
      for design in DESIGNS for scenario in SCENARIOS]
val_results=pd.DataFrame(rows)
display(val_results[['design','scenario','optimal_easy_fraction',
                     'optimal_medium_fraction','optimal_hard_fraction',
                     'oracle_minus_best_fixed','best_fixed_action']])
val_results.to_csv(WORK/'validation_sensitivity.csv',index=False)

In [ ]:
# A design is frozen by a deterministic, *validation-only* diagnostic rule:
# prefer action diversity under moderate assumptions, then small oracle-vs-fixed gap
# is NOT rewarded; instead use an explicit, stable default if none qualifies.
eligible=val_results[val_results.scenario=='moderate'].copy()
eligible['nontrivial']=1-eligible[
 ['optimal_easy_fraction','optimal_medium_fraction','optimal_hard_fraction']].max(axis=1)
# Rule predeclared: balanced utility unless its action diversity is <5%;
# if so, pick the first of mastery_focus/suitability with >=5%; otherwise retain balanced and flag.
chosen='balanced'
if float(eligible.loc[eligible.design=='balanced','nontrivial'].iloc[0])<0.05:
    for candidate in ['mastery_focus','suitability']:
        if float(eligible.loc[eligible.design==candidate,'nontrivial'].iloc[0])>=0.05:
            chosen=candidate
            break
print('Selected for LOCKED TEST audit:',chosen)
print('Selection based only on validation simulation; no causal tuning.')

### Locked test audit

The held-out test split is used only **after the design has been selected**. This reports action diversity, oracle versus fixed-policy reward, and scenario sensitivity **inside the assumed simulator**.

In [ ]:
test=pred[pred.split=='test'].copy()
test_rows=[summarize(test.p_observed.to_numpy(),'test',chosen,scenario)
           for scenario in SCENARIOS]
test_results=pd.DataFrame(test_rows)
display(test_results)
test_results.to_csv(WORK/'locked_test_sensitivity.csv',index=False)

In [ ]:
# Readiness groups are defined by p_observed (observable predictive score), not true mastery.
bins=[0,.4,.7,1.000001]
labels=['low (<0.4)','medium (0.4-0.7)','high (>=0.7)']
records=[]
for scenario,d in SCENARIOS.items():
    q,u=utilities(test.p_observed.to_numpy(),d,chosen)
    opt=u.argmax(axis=1)
    g=pd.cut(test.p_observed,bins=bins,labels=labels,include_lowest=True,right=False)
    for label in labels:
        mask=(g==label).to_numpy()
        if not mask.any():continue
        records.append({'scenario':scenario,'readiness_group':label,'n':int(mask.sum()),
           **{f'fraction_{a}':float((opt[mask]==i).mean()) for i,a in enumerate(ACTIONS)}})
group_results=pd.DataFrame(records)
display(group_results)
group_results.to_csv(WORK/'readiness_action_choices.csv',index=False)

In [ ]:
# Plot action choices across predictive readiness without accessing held-out outcomes.
grid=np.linspace(.01,.99,99)
fig,axs=plt.subplots(1,2,figsize=(12,4))
for ax,scenario in zip(axs,['zero','moderate']):
    q,u=utilities(grid,SCENARIOS[scenario],chosen)
    for i,a in enumerate(ACTIONS): ax.plot(grid,u[:,i],label=a)
    ax.set(title=f'{chosen}: {scenario} assumed effect',xlabel='Observed-context predicted correctness',ylabel='Assumed expected utility')
    ax.legend()
fig.tight_layout()
fig.savefig(WORK/'utility_curves.png',dpi=170)
plt.show()

### Cluster bootstrap uncertainty

Bootstrap learners, not individual rows. These intervals measure **sampling variability conditional on the simulator assumptions**; they do not include uncertainty about causal action effects.

In [ ]:
def cluster_bootstrap(df,design,scenario,n_boot=300,seed=42):
    gen=np.random.default_rng(seed)
    ids=df.user_id.unique()
    grouped=df.groupby('user_id').indices
    p=df.p_observed.to_numpy()
    _,u=utilities(p,SCENARIOS[scenario],design)
    best=u.max(axis=1)
    means=[]
    for _ in range(n_boot):
        sampled=gen.choice(ids,size=len(ids),replace=True)
        idx=np.concatenate([grouped[x] for x in sampled])
        sub=u[idx]
        means.append(float(sub.max(axis=1).mean()-sub.mean(axis=0).max()))
    return {'scenario':scenario,'design':design,'bootstrap_reps':n_boot,
            'oracle_minus_best_fixed_ci_low':float(np.quantile(means,.025)),
            'oracle_minus_best_fixed_median':float(np.median(means)),
            'oracle_minus_best_fixed_ci_high':float(np.quantile(means,.975))}
boot=pd.DataFrame([cluster_bootstrap(test,chosen,s) for s in SCENARIOS])
display(boot)
boot.to_csv(WORK/'learner_cluster_bootstrap.csv',index=False)

In [ ]:
# Compare old original-like utility with redesign using identical assumed effects.
comparison=[]
for scenario in SCENARIOS:
    for design in ['original_like',chosen]:
        comparison.append(summarize(test.p_observed.to_numpy(),'test',design,scenario))
comparison=pd.DataFrame(comparison)
comparison.to_csv(WORK/'original_vs_redesigned.csv',index=False)
display(comparison[['design','scenario','optimal_easy_fraction','optimal_medium_fraction',
                    'optimal_hard_fraction','oracle_minus_best_fixed']])

In [ ]:
# Explicit limitations and quality gates
mod=test_results[test_results.scenario=='moderate'].iloc[0]
diversity=1-max(mod[f'optimal_{a}_fraction'] for a in ACTIONS)
report={
 'stage':'4A-C',
 'input_stage4a_spec':spec,
 'input_stage4ab_audit':audit,
 'selected_design':chosen,
 'designs':DESIGNS,
 'assumed_effect_scenarios':SCENARIOS,
 'challenge_credits':CHALLENGE.tolist(),
 'assumed_costs':COST.tolist(),
 'validation_only_selection':True,
 'locked_test_audit':True,
 'moderate_test_action_diversity':float(diversity),
 'moderate_test_oracle_minus_best_fixed':float(mod.oracle_minus_best_fixed),
 'nontrivial_simulated_policy_at_moderate_effect':bool(diversity>=0.05),
 'causal_effects_identified':False,
 'actual_learning_gain_established':False,
 'sequential_RL_ready':False,
 'interpretation':'All reward improvements are in-model assumed utility; no causal or longitudinal claims.'
}
(WORK/'stage4ac_report.json').write_text(json.dumps(report,indent=2))
print('Moderate scenario nontrivial diversity >=5%:',report['nontrivial_simulated_policy_at_moderate_effect'])
print('No real-world learning gain or causal action effect is established.')

In [ ]:
readme='''SHIVA-LLM Objective 02 Stage 4A-C
Input: Stage 4A simulator and Stage 4A-B robustness audit ZIPs.
Output: validation sensitivity, frozen-design test audit, readiness choices,
learner bootstrap intervals, utility plots, and full assumptions report.

All actions and challenge credits are synthetic design assumptions.
The model is a one-step contextual bandit, NOT a validated sequential RL environment.
No real-world instructional effect is identified.
'''
(WORK/'README.txt').write_text(readme)
zip_path=pathlib.Path('/content/objective02_stage4ac_reward_redesign.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(WORK.iterdir()):
        if f.is_file():z.write(f,arcname=f.name)
print('Created:',zip_path, 'bytes:',zip_path.stat().st_size)
files.download(str(zip_path))